<a href="https://colab.research.google.com/github/dev-manitb/Distorted-Visual-Sequence-Pattern-Recognition-using-Deep-Learning/blob/main/notebook_ManitBansal_24115100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!unzip -q cig_ps.zip

In [3]:
# Quick install for our evaluation metric
!pip install Levenshtein -q

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import pandas as pd
from PIL import Image
from tqdm import tqdm  # Added this for awesome progress bars!

# ---------------------------------------------------------
# 1. Setup & Hyperparameters (The tweakable stuff)
# ---------------------------------------------------------
# Using lower-case here because I adjust these manually all the time
batch_size = 32
learning_rate = 0.0001
num_epochs = 30
image_h = 64
image_w = 256

# Check if we got the sweet Colab GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Vocab setup (0-9, A-Z). We leave index 0 empty for the CTC 'blank' token
vocab = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
char_to_int = {char: i + 1 for i, char in enumerate(vocab)}
int_to_char = {i: char for char, i in char_to_int.items()}
num_classes = len(vocab) + 1


# ---------------------------------------------------------
# 2. Loading the Dataset
# ---------------------------------------------------------
class CaptchaDataset(Dataset):
    def __init__(self, csv_file, img_folder, transform=None, is_test=False):
        # If it's the test set, we don't have a CSV, just a list of files
        if is_test:
            self.data = pd.DataFrame({'image': os.listdir(img_folder)})
        else:
            self.data = pd.read_csv(csv_file)

        self.img_folder = img_folder
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_filename = self.data.iloc[idx]['image']
        img_path = os.path.join(self.img_folder, img_filename)

        # Open and force it to grayscale
        img = Image.open(img_path).convert('L')

        if self.transform:
            img = self.transform(img)

        if self.is_test:
            return img, img_filename

        # Grab the text, make it uppercase just in case, and map to numbers
        raw_text = str(self.data.iloc[idx]['text']).upper()
        encoded_text = [char_to_int[c] for c in raw_text if c in char_to_int]

        return img, torch.tensor(encoded_text, dtype=torch.long)

def custom_collate(batch):
    """
    Groups a list of samples into a batch.
    Crucial for CTC Loss since our text labels aren't all the same length.
    """
    images, labels = zip(*batch)
    images = torch.stack(images, 0)

    # CTC needs to know exactly how long each label was before we squashed them together
    target_lengths = torch.tensor([len(label) for label in labels], dtype=torch.long)
    targets = torch.cat(labels)

    return images, targets, target_lengths


# ---------------------------------------------------------
# 3. The Model (CRNN)
# ---------------------------------------------------------
class MyCRNN(nn.Module):
    def __init__(self, classes, hidden_size=256):
        super(MyCRNN, self).__init__()

        # CNN to find the shapes and edges
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(),
            # Squish height, keep width so the RNN can read left-to-right
            nn.MaxPool2d((2, 1), (2, 1)),

            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2, 1), (2, 1)),

            nn.Conv2d(512, 512, 2, 1, 0), nn.BatchNorm2d(512), nn.ReLU()
        )

        # Linear bridge
        self.bridge = nn.Linear(512 * 3, hidden_size)

        # RNN to figure out the sequence of those shapes
        self.rnn = nn.LSTM(hidden_size, hidden_size, bidirectional=True, num_layers=2, dropout=0.3)
        self.classifier = nn.Linear(hidden_size * 2, classes)

    def forward(self, x):
        features = self.cnn(x)

        # I left this here because it took me forever to figure out the math for the Linear layer!
        # print("CNN Output shape:", features.shape)

        # Reshape: (Batch, Channels, Height, Width) -> (Width, Batch, Features)
        batch_sz, channels, height, width = features.size()
        features = features.view(batch_sz, channels * height, width)
        features = features.permute(2, 0, 1)

        seq_features = self.bridge(features)
        rnn_out, _ = self.rnn(seq_features)
        output = self.classifier(rnn_out)

        return output


# ---------------------------------------------------------
# 4. Decoding & Training Logic
# ---------------------------------------------------------
def decode_predictions(outputs):
    """Turns the model's raw probability math back into ABC/123 strings."""
    # Grab the most likely character index for each time step
    best_guesses = torch.argmax(outputs, dim=-1).permute(1, 0)
    final_strings = []

    for row in best_guesses:
        chars = []
        prev_char = None
        for c in row:
            val = c.item()
            # Ignore blanks (0) and ignore duplicates (e.g., 'HHH' becomes 'H')
            if val != 0 and val != prev_char:
                chars.append(int_to_char[val])
            prev_char = val

        final_strings.append("".join(chars))

    return final_strings


def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0

    # Adding tqdm here so we get a nice progress bar while we wait!
    loop = tqdm(dataloader, leave=False)
    for images, targets, target_lengths in loop:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        preds = model(images)

        pred_lengths = torch.full(size=(preds.size(1),), fill_value=preds.size(0), dtype=torch.long).to(device)
        loss = criterion(preds.log_softmax(2), targets, pred_lengths, target_lengths)

        loss.backward()
        # The lifesaver line: without clipping, the model explodes into a mode collapse
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)


# ---------------------------------------------------------
# 5. Let's run it!
# ---------------------------------------------------------
if __name__ == "__main__":
    # Figure out where Colab hid our files
    base_folder = "cig_ps/" if os.path.exists("cig_ps/train-labels.csv") else ""
    if not os.path.exists(os.path.join(base_folder, "train-labels.csv")):
        base_folder = "cig_data/" if os.path.exists("cig_data/train-labels.csv") else ""

    train_csv = os.path.join(base_folder, "train-labels.csv")
    train_dir = os.path.join(base_folder, "train_images")
    test_dir = os.path.join(base_folder, "test_images")

    if os.path.exists(train_csv):
        print("Found the data! Setting up datasets...")

        # Simple transforms. Removed ColorJitter because the dataset is already messy enough.
        img_transforms = transforms.Compose([
            transforms.Resize((image_h, image_w)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

        train_data = CaptchaDataset(train_csv, train_dir, transform=img_transforms)
        test_data = CaptchaDataset(None, test_dir, transform=img_transforms, is_test=True)

        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=custom_collate)
        test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

        # Build model & tools
        model = MyCRNN(classes=num_classes).to(device)
        loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate)

        print("\n--- Starting Training ---")
        for epoch in range(1, num_epochs + 1):
            avg_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
            print(f"Epoch {epoch}/{num_epochs} Complete! Average Loss: {avg_loss:.4f}")

        # --- Inference / Creating the Submission ---
        print("\nTraining done! Looking at the test set now...")
        model.eval()
        test_images_names = []
        test_predictions = []

        with torch.no_grad():
            for images, filenames in tqdm(test_loader, desc="Predicting"):
                images = images.to(device)
                preds = model(images)
                decoded = decode_predictions(preds)

                test_images_names.extend(filenames)
                test_predictions.extend(decoded)

        # Format the CSV exactly how the professors asked for it
        submission_df = pd.DataFrame({'image': test_images_names, 'prediction': test_predictions})
        filename = "submission_ManitBansal_24115100.csv"
        submission_df.to_csv(filename, index=False)

        print(f"\nSuccess! Wrote everything to {filename}.")
        print("Check the folder icon on the left to download it.")

    else:
        print(f"Uh oh, couldn't find {train_csv}. Make sure you unzipped everything properly!")

Running on: cuda
Found the data! Setting up datasets...

--- Starting Training ---


Epoch 1/30 Complete! Average Loss: 3.7613


Epoch 2/30 Complete! Average Loss: 3.5716


Epoch 3/30 Complete! Average Loss: 3.5698


Epoch 4/30 Complete! Average Loss: 3.5684


...

Epoch 28/30 Complete! Average Loss: 0.0074


Epoch 29/30 Complete! Average Loss: 0.0057


Epoch 30/30 Complete! Average Loss: 0.0067

Training done! Looking at the test set now...


Predicting: 100%|██████████| 157/157 [00:13<00:00, 11.67it/s]


Success! Wrote everything to submission_ManitBansal_24115100.csv.
Check the folder icon on the left to download it.
